In [1]:
from collections import deque

In [2]:
def read_problems(filename):
    with open(filename) as f:
        lines = f.readlines()

    problems = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith("problem"):
            problem = {}
            problem["id"] = line.split()[1]

            size = lines[i + 1].split()
            problem["rows"] = int(size[1])
            problem["cols"] = int(size[2])

            exit_line = lines[i + 2].split()
            exit_cols = []
            for x in exit_line[2:]:
                exit_cols.append(int(x))
            problem["exit"] = exit_cols

            problem["target"] = lines[i + 3].split()[1]

            board = []
            for r in range(problem["rows"]):
                board.append(tuple(lines[i + 5 + r].split()))
            problem["board"] = tuple(board)

            problems.append(problem)
            i = i + 5 + problem["rows"] + 1
        else:
            i += 1
    return problems


problems = read_problems("klotski_problems.txt")
print("number of problems:", len(problems))
print(problems[0])

number of problems: 6
{'id': 'p01', 'rows': 3, 'cols': 3, 'exit': [1], 'target': 'A', 'board': (('A', '5', '1'), ('6', '4', '2'), ('3', '.', '.'))}


In [3]:
def print_board(board):
    for row in board:
        line = ""
        for cell in row:
            line += cell.rjust(3)
        print(line)


print_board(problems[1]["board"])

  A  A  4  7
  A  A  6  8
  1  5  .  3
 10  .  9  2


In [4]:
def find_pieces(board):
    pieces = {}
    for r in range(len(board)):
        for c in range(len(board[0])):
            name = board[r][c]
            if name != ".":
                if name not in pieces:
                    pieces[name] = []
                pieces[name].append((r, c))
    return pieces


print(find_pieces(problems[2]["board"]))

{'A': [(0, 0), (0, 1), (1, 0), (1, 1)], '2': [(0, 2)], '3': [(0, 3)], '5': [(1, 2)], 'B': [(2, 0), (3, 0)], '1': [(2, 1)], 'C': [(2, 2), (2, 3)], '6': [(3, 2)], '4': [(3, 3)]}


In [5]:
def is_goal(board, target, exit_cols):
    last_row = board[len(board) - 1]
    for c in exit_cols:
        if last_row[c] != target:
            return False
    return True

In [6]:
DIRECTIONS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}

def get_next_boards(board):
    rows = len(board)
    cols = len(board[0])
    pieces = find_pieces(board)
    result = []

    for name in pieces:
        cells = pieces[name]
        for direction in DIRECTIONS:
            dr, dc = DIRECTIONS[direction]

            can_move = True
            for (r, c) in cells:
                nr = r + dr
                nc = c + dc
                if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
                    can_move = False
                    break
                if board[nr][nc] != "." and board[nr][nc] != name:
                    can_move = False
                    break

            if can_move:
                new_board = []
                for row in board:
                    new_board.append(list(row))
                for (r, c) in cells:
                    new_board[r][c] = "."
                for (r, c) in cells:
                    new_board[r + dr][c + dc] = name

                new_board_tuple = []
                for row in new_board:
                    new_board_tuple.append(tuple(row))
                result.append((tuple(new_board_tuple), (name, direction)))

    return result


for new_board, move in get_next_boards(problems[0]["board"]):
    print(move)

('4', 'down')
('2', 'down')
('3', 'right')


In [7]:
def make_key(board, target):
    pieces = find_pieces(board)
    key = []
    for row in board:
        key.append(list(row))

    for name in pieces:
        if name == target:
            continue
        piece_rows = []
        piece_cols = []
        for (r, c) in pieces[name]:
            piece_rows.append(r)
            piece_cols.append(c)
        height = max(piece_rows) - min(piece_rows) + 1
        width = max(piece_cols) - min(piece_cols) + 1
        for (r, c) in pieces[name]:
            key[r][c] = str(height) + "x" + str(width)

    key_tuple = []
    for row in key:
        key_tuple.append(tuple(row))
    return tuple(key_tuple)


print_board(make_key(problems[2]["board"], "A"))

  A  A1x11x1
  A  A1x1  .
2x11x11x21x2
2x1  .1x11x1


In [8]:
def bfs(start, target, exit_cols):
    queue = deque()
    queue.append(start)
    parent = {make_key(start, target): None}

    while len(queue) > 0:
        board = queue.popleft()

        if is_goal(board, target, exit_cols):
            moves = []
            while parent[make_key(board, target)] is not None:
                previous, move = parent[make_key(board, target)]
                moves.append(move)
                board = previous
            moves.reverse()
            return moves, len(parent)

        for new_board, move in get_next_boards(board):
            key = make_key(new_board, target)
            if key not in parent:
                parent[key] = (board, move)
                queue.append(new_board)

    return None, len(parent)

In [9]:
for p in problems:
    print("problem", p["id"])
    print_board(p["board"])

    moves, visited = bfs(p["board"], p["target"], p["exit"])

    if moves is None:
        print("no solution, visited states:", visited)
    else:
        print("solved in", len(moves), "moves, visited states:", visited)
        text = []
        for (piece, direction) in moves:
            text.append(piece + " " + direction)
        print("moves:", ", ".join(text))
    print()

problem p01
  A  5  1
  6  4  2
  3  .  .
solved in 8 moves, visited states: 143
moves: 4 down, 5 down, A right, 6 up, 5 left, A down, 4 right, A down

problem p02
  A  A  4  7
  A  A  6  8
  1  5  .  3
 10  .  9  2
solved in 18 moves, visited states: 266
moves: 5 down, 1 right, 1 right, A down, 4 left, 4 left, 6 up, 6 left, 1 up, 1 up, A right, 10 up, 10 up, 5 left, 5 up, 9 left, 9 left, A down

problem p03
  A  A  2  3
  A  A  5  .
  B  1  C  C
  B  .  6  4
solved in 45 moves, visited states: 2141
moves: 3 down, 2 right, 1 down, C left, 3 down, 5 right, A right, B up, B up, C left, 6 up, 1 right, C down, 6 left, 6 left, A down, 2 left, 2 left, 5 up, 5 left, 3 up, 3 up, A right, 2 down, 2 down, B right, 6 up, 6 up, 2 left, 2 up, C up, 1 left, 1 left, 4 left, 4 left, A down, 5 down, 5 right, B right, 6 right, 2 up, C up, 1 up, 4 left, A left

problem p04
  B  A  A  C
  B  A  A  C
  D  F  F  E
  D  2  3  E
  1  .  .  4
solved in 116 moves, visited states: 24099
moves: 1 right, D down, F